Strategia inwestycyjna (decyzje wejścia i wyjścia) dla spółki  Apple (AAPL), test w okresie od 01.01.2024 - 06.05.2024

In [1]:
pip install ta

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=173488922d0a089eb1e90c1ebcb2678fd75a62395087f75d6f14a4fcb8d0e47b
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from ta.momentum import RSIIndicator
from ta.trend import SMAIndicator
from ta.volatility import AverageTrueRange

# Pobranie danych giełdowych AAPL od 2020 roku
ticker = "AAPL"
start = "2020-01-01"
end   = "2024-05-07"

data = yf.download(ticker, start=start, end=end)
data.dropna(inplace=True)

data.columns = data.columns.get_level_values(0)
data.columns = ["Open", "High", "Low", "Close", "Volume"]

#Wskaźniki
# RSI - momentum
rsi_period = 14
data["RSI"] = RSIIndicator(close=data["Close"], window=rsi_period).rsi()

# SMA_20 - śr. krocząca
sma_window = 20
data["SMA_20"] = SMAIndicator(close=data["Close"], window=sma_window).sma_indicator()

# ATR - do SL/TP
atr_period = 14
atr = AverageTrueRange(
    high=data["High"],
    low=data["Low"],
    close=data["Close"],
    window=atr_period
)
data["ATR"] = atr.average_true_range()

# Lag1 Close - opóźnienie cen close
data["Close_lag1"] = data["Close"].shift(1)


data.dropna(inplace=True)
print(data.columns)

data = data.copy()
data.columns = [str(c) for c in data.columns]  # bez multindexu

data

/tmp/ipykernel_32200/1257164004.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'RSI', 'SMA_20', 'ATR',
       'Close_lag1'],
      dtype='object')


                  Open        High         Low       Close     Volume  \
Date                                                                    
2020-01-30   77.998230   78.051213   76.765172   77.196263  126743200   
2020-01-31   74.539902   77.711655   74.246086   77.290199  199588400   
2020-02-03   74.335190   75.498405   72.784231   73.285159  173788400   
2020-02-04   76.789268   76.979528   75.532125   75.936721  136616400   
2020-02-05   77.415428   78.212581   76.813348   77.913945  118826800   
...                ...         ...         ...         ...        ...   
2024-04-30  168.638458  173.252184  168.311734  171.608665   65934800   
2024-05-01  167.618668  170.994806  167.430552  167.895886   50383100   
2024-05-02  171.311646  171.697772  169.192898  170.796805   94214900   
2024-05-03  181.558899  185.142945  180.846048  184.796414  163224100   
2024-05-06  179.905441  182.370703  178.628244  180.539085   78569700   

                  RSI      SMA_20       ATR Close_

In [3]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error
from itertools import product


In [4]:

# Przygotowanie danych dla Prophet
df = data.copy()
df_prophet = pd.DataFrame({
    "ds": df.index,
    "y": df["Close"],
    "RSI": df["RSI"],
    "SMA_20": df["SMA_20"],
    "ATR": df["ATR"],
    "Close_lag1": df["Close_lag1"]
})
df_prophet["ds"] = pd.to_datetime(df_prophet["ds"])
print(df_prophet)

# Podział na zbiór test i train
train_df = df_prophet[df_prophet["ds"] < "2024-01-01"]
test_df_prophet = df_prophet[df_prophet["ds"] >= "2024-01-01"]


# Siatka hiperparametrów
cp_scales = [0.01, 0.05, 0.1]
season_scales = [1.0, 5.0, 10.0]

best_params = None
best_mae = np.inf

#Przeszukiwanie siatki hiperparametrów
for cp, ss in product(cp_scales, season_scales):
    m = Prophet(
        changepoint_prior_scale=cp,
        seasonality_prior_scale=ss,
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=False
    )
    # dodatek regresorów
    for reg in ["RSI", "SMA_20", "ATR", "Close_lag1"]:
        m.add_regressor(reg)

#trenowanie na danych hist
    m.fit(train_df)

    future = train_df[["ds", "RSI", "SMA_20", "ATR", "Close_lag1"]].copy()
    forecast = m.predict(future)

#MAE
    mae = mean_absolute_error(train_df["y"], forecast["yhat"])
    if mae < best_mae:
        best_mae = mae
        best_params = {"changepoint_prior_scale": cp, "seasonality_prior_scale": ss}

print("Najlepsze parametry:", best_params, "MAE:", best_mae)


                   ds           y        RSI      SMA_20       ATR  Close_lag1
Date                                                                          
2020-01-30 2020-01-30   77.196263  62.968872   74.822392  1.728734   78.137938
2020-01-31 2020-01-31   77.290199  63.311197   75.119700  1.852794   77.196263
2020-02-03 2020-02-03   73.285159  44.445871   75.205797  2.042306   77.290199
2020-02-04 2020-02-04   75.936721  54.180397   75.464933  2.160310   73.285159
2020-02-05 2020-02-05   77.913945  59.832520   75.750078  2.168564   75.936721
...               ...         ...        ...         ...        ...         ...
2024-04-30 2024-04-30  171.608665  56.272435  167.929566  3.882128  171.648248
2024-05-01 2024-05-01  167.895886  47.384969  167.968673  3.903270  171.608665
2024-05-02 2024-05-02  170.796805  53.556945  168.078571  3.896029  167.895886
2024-05-03 2024-05-03  184.796414  71.147078  168.923102  4.642465  170.796805
2024-05-06 2024-05-06  180.539085  63.295978  169.5

In [5]:
#model z najlepszymi hiperparametrami
m_best = Prophet(
    changepoint_prior_scale=best_params["changepoint_prior_scale"],
    seasonality_prior_scale=best_params["seasonality_prior_scale"],
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=False
)
for reg in ["RSI", "SMA_20", "ATR", "Close_lag1"]:
    m_best.add_regressor(reg)

m_best.fit(train_df)

# Przygotowanie danych testowych
future_test = test_df_prophet[["ds", "RSI", "SMA_20", "ATR", "Close_lag1"]]
forecast_test = m_best.predict(future_test)
df.loc[test_df_prophet["ds"], "yhat"] = forecast_test["yhat"].values


In [6]:
threshold = 0.001

df["yhat_shift1"] = df["yhat"].shift(1)  # prognoza na dzisiejszy dzień wygenerowana wczoraj
df["signal"] = 0

#wejście w longa
long_cond = (
    (df["yhat_shift1"] > df["Close"] * (1 + threshold)) &
    (df["Close"] > df["SMA_20"]) &
    (df["RSI"] < 70)
)

df.loc[long_cond, "signal"] = 1


In [7]:
test_start = "2023-12-31"
test_end   = "2024-05-07"
test_df = df.loc[test_start:test_end].copy()
test_df

                  Open        High         Low       Close     Volume  \
Date                                                                    
2024-01-02  183.562134  186.330796  181.831722  185.055227   82488700   
2024-01-03  182.187775  183.799536  181.376945  182.158112   58414500   
2024-01-04  179.873932  181.040717  178.855462  180.111236   71983600   
2024-01-05  179.152115  180.714432  178.153425  179.953062   62379700   
2024-01-08  183.483063  183.522623  179.468508  180.051901   59144500   
...                ...         ...         ...         ...        ...   
2024-04-30  168.638458  173.252184  168.311734  171.608665   65934800   
2024-05-01  167.618668  170.994806  167.430552  167.895886   50383100   
2024-05-02  171.311646  171.697772  169.192898  170.796805   94214900   
2024-05-03  181.558899  185.142945  180.846048  184.796414  163224100   
2024-05-06  179.905441  182.370703  178.628244  180.539085   78569700   

                  RSI      SMA_20       ATR Close_

In [8]:
pip install backtesting

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 5.0 MB/s eta 0:00:00


In [9]:
# TRAIN: 2020–2023
train_data = data.loc['2020-01-01':'2023-12-31'].copy()
train_data['Prediction'] = df.loc['2020-01-01':'2023-12-31', 'yhat']

# TEST: 2024
test_data = data.loc['2024-01-01':'2024-12-31'].copy()
test_data['Prediction'] = df.loc['2024-01-01':'2024-12-31', 'yhat']


In [10]:
from backtesting import Strategy, Backtest
class HighWinRateStrategyy(Strategy):
    tp_atr = 0.6
    sl_atr = 1.2
    threshold = 0.001
    rsi_upper = 60
    rsi_lower = 40
    atr_filter = 0.7

    def init(self):
      self.pred = self.I(lambda: self.data.Prediction)
      self.atr = self.I(lambda: self.data.ATR)
      self.rsi = self.I(lambda: self.data.RSI)
      self.atr_median = np.median(self.data.ATR)

    def next(self):
        price = self.data.Close[-1]
        vol = self.atr[-1]

        #brak handlu przy niskim ATR
        if vol < self.atr_median * self.atr_filter:
            return

        # Rsi ok. 50 - nie handluj
        if 45 < self.rsi[-1] < 55:
            return

        # ignoruj płaskie sygnały
        if abs(self.pred[-1] - price) < price * 0.001:
            return

        sl_dist = vol * self.sl_atr
        tp_dist = vol * self.tp_atr

        signal_long = (
            self.pred[-1] > price * (1 + self.threshold)
            and self.rsi[-1] < self.rsi_upper
        )

        signal_short = (
            self.pred[-1] < price * (1 - self.threshold)
            and self.rsi[-1] > self.rsi_lower
        )

        #wyjście
        if signal_short and self.position.is_long:
            self.position.close()
        if signal_long and self.position.is_short:
            self.position.close()

        # Wejścia
        if signal_long and not self.position:
            self.buy(sl=price - sl_dist, tp=price + tp_dist)
        elif signal_short and not self.position:
            self.sell(sl=price + sl_dist, tp=price - tp_dist)


/usr/local/lib/python3.12/dist-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


In [11]:
bt_train = Backtest(train_data, HighWinRateStrategyy, cash=10000, commission=0.001)

stats_train, heatmap_train = bt_train.optimize(
    tp_atr=[0.4, 0.5, 0.6, 0.7, 0.8],
    sl_atr=[1.0, 1.2, 1.4],
    threshold=[0.0010, 0.0015, 0.0020, 0.0025],
    rsi_upper=[45, 50, 55, 60],
    rsi_lower=[30, 35, 40],
    atr_filter=[0.5, 0.6, 0.7, 0.8],

    maximize='Return [%]',
    constraint=lambda p: p.tp_atr <= p.sl_atr * 1.5,
    return_heatmap=True
)

print("NAJLEPSZE PARAMETRY (TRAIN)")
print(stats_train._strategy)


/usr/local/lib/python3.12/dist-packages/backtesting/backtesting.py:1624: UserWarning: Searching for best of 2880 configurations.
  output = _optimize_grid()



--- NAJLEPSZE PARAMETRY (TRAIN) ---
HighWinRateStrategyy(tp_atr=0.4,sl_atr=1.0,threshold=0.001,rsi_upper=45,rsi_lower=30,atr_filter=0.5)


In [12]:
best_params = stats_train._strategy

bt_test = Backtest(test_data, HighWinRateStrategyy, cash=10000, commission=0.001)

strategy_params = {
    'tp_atr': best_params.tp_atr,
    'sl_atr': best_params.sl_dist if hasattr(best_params, 'sl_dist') else best_params.sl_atr,
    'threshold': best_params.threshold,
    'rsi_upper': best_params.rsi_upper,
    'rsi_lower': best_params.rsi_lower,
    'atr_filter': best_params.atr_filter
}

stats_test = bt_test.run(**strategy_params)

print("wyniki na test")
print(stats_test)

wyniki na test
Start                      2024-01-02 00:00:00
End                        2024-05-06 00:00:00
Duration                     125 days 00:00:00
Exposure Time [%]                     59.77011
Equity Final [$]                   10888.35727
Equity Peak [$]                     11091.2187
Commissions [$]                      560.77301
Return [%]                             8.88357
Buy & Hold Return [%]                 -2.44043
Return (Ann.) [%]                     27.95681
Volatility (Ann.) [%]                 16.99971
CAGR [%]                              18.71788
Sharpe Ratio                          1.64455
Sortino Ratio                         4.28518
Calmar Ratio                          8.05295
Alpha [%]                             9.03907
Beta                                  0.06372
Max. Drawdown [%]                     -3.47162
Avg. Drawdown [%]                        -1.23
Max. Drawdown Duration       32 days 00:00:00
Avg. Drawdown Duration       11 days 00:00:00
# Tra

W badanym okresie strategia osiągnęła stopę zwrotu +8.88%, podczas gdy proste kup i trzymaj (Buy & Hold) dla AAPL wygenerowało wynik –2.44%. Oznacza to, że strategia pokonała rynek o ponad 11 punktów procentowych.

Strategia wykonała 27 transakcji, co odpowiada jej charakterowi swingowemu. Nie jest nadmiernie aktywna, ale reaguje na wyraźne sygnały z modelu predykcyjnego i wskaźników technicznych.

Strategia wykazuje wysoki poziom bezpieczeństwa kapitału, potwierdzony niskim maksymalnym obsunięciem (Max Drawdown -3.48%) oraz wysokim wskaźnikiem Profit Factor (2.23).